# EODH Universe, Weighting, and Inference Audit

## Purpose and boundary

This notebook reviews the official EODH universe, trip reference period, weighting approach, geographic coverage, and published aggregate benchmarks before any OD matrix is constructed. It reads documentation and the existing schema/catalog only. It does not read microdata rows, apply `fexp_vj`, or generate matrices.

The EODH technical sheet describes model-based inference using a doubly robust estimator, integration with EODH 2019, and calibration to 2023 household and person population projections. Therefore, `fexp_vj` must not be assumed to be a conventional design weight, and conventional design-based standard errors must not be calculated without further evidence. The operational relationship between this documented methodology and the trip-level `fexp_vj` field remains to be verified.

The purpose of this review is to establish an evidence ledger and a decision gate. No sample or expanded matrix is produced here.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'data/sources/raw').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the notebook path.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import pandas as pd
from pypdf import PdfReader

OFFICIAL_SHEET = Path('data/sources/raw/Datos tecnicos de la Encuesta/a. Ficha tecnica .pdf')
PROCESSING_REPORT = Path('data/sources/raw/contrato 2022-1972/Etapa IV/P9_INFORME FINAL ETAPA IV.pdf')
INDICATOR_REPORT = Path('data/sources/raw/documentation/a. Estimacion final de indicadores.pdf')
for relative_path in (OFFICIAL_SHEET, PROCESSING_REPORT, INDICATOR_REPORT):
    if not (PROJECT_ROOT / relative_path).is_file():
        raise FileNotFoundError(f'Required methodology source is missing: {relative_path.as_posix()}')

print(f'Project root: {PROJECT_ROOT}')
print('Methodology sources are present; only document metadata and cited pages will be reviewed.')

Project root: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect
Methodology sources are present; only document metadata and cited pages will be reviewed.


## 1. Documented EODH evidence

The evidence below separates official survey specifications from contextual processing reports and published result benchmarks. The benchmark total is a comparison target only: it can be used to validate a reproduced matrix only after its population, trip eligibility, geography, and mode treatment are shown to match.

In [2]:
evidence = pd.DataFrame([
    {'evidence_id': 'EODH-01', 'topic': 'Target population and geography', 'documented_finding': 'Private households with residents aged 5 or older in urban and rural Bogota and urban areas of 20 neighboring municipalities.', 'source_path': OFFICIAL_SHEET.as_posix(), 'page': 20, 'evidence_status': 'documented_official'},
    {'evidence_id': 'EODH-02', 'topic': 'Statistical units', 'documented_finding': 'The observation unit is persons in sampled households; the analysis units include households, persons, socioeconomic level, and UTAM; the sampling unit is an average measure of about ten dwellings.', 'source_path': OFFICIAL_SHEET.as_posix(), 'page': 20, 'evidence_status': 'documented_official'},
    {'evidence_id': 'EODH-03', 'topic': 'Travel reference period', 'documented_finding': 'Travel behavior refers to the day before the interview; the Stage IV report specifies the preceding non-holiday weekday (Monday to Friday).', 'source_path': PROCESSING_REPORT.as_posix(), 'page': 13, 'evidence_status': 'documented_official_and_processing_report'},
    {'evidence_id': 'EODH-04', 'topic': 'Weighting and inference approach', 'documented_finding': 'The technical sheet describes a doubly robust model-based estimator integrating EODH 2019 and calibration to 2023 household and person population projections. Fieldwork adjustments and selection/inclusion bias were reported as reasons to use this alternative to inference based only on the planned sample design.', 'source_path': OFFICIAL_SHEET.as_posix(), 'page': 11, 'evidence_status': 'documented_official'},
    {'evidence_id': 'EODH-05', 'topic': 'Final analytical record counts', 'documented_finding': 'The final EODH database reports 22,755 households, 67,556 persons, 48,539 persons with trips, 100,174 trips, and 122,992 stages.', 'source_path': PROCESSING_REPORT.as_posix(), 'page': 127, 'evidence_status': 'published_aggregate_benchmark'},
    {'evidence_id': 'EODH-06', 'topic': 'Published expanded trip total', 'documented_finding': 'The report gives 16,723,044 daily trips, with margin of error 446,648, estimated CV 1.36%, and confidence interval 16,276,395 to 17,169,692 for the reported study population.', 'source_path': PROCESSING_REPORT.as_posix(), 'page': 128, 'evidence_status': 'published_aggregate_benchmark'},
])

pending = pd.DataFrame([
    {'decision_id': 'PENDING-01', 'topic': 'Trip eligibility', 'evidence_needed': 'Map the official trip reference population and valid/complete EODH records to the released viajes table; specify treatment of unclassified and missing fields.'},
    {'decision_id': 'PENDING-02', 'topic': 'Trip expansion factor', 'evidence_needed': 'Find the field-level definition and operational derivation of fexp_vj; verify it is the final model-based trip expansion factor and identify its valid universe.'},
    {'decision_id': 'PENDING-03', 'topic': 'ZAT geography', 'evidence_needed': 'Verify zat_ori and zat_des code domains and the crosswalk to the authoritative EODH study-area ZAT geography, including outside and unresolved codes.'},
    {'decision_id': 'PENDING-04', 'topic': 'Mode dimension', 'evidence_needed': 'Confirm the published definition and domains of modo_principal_agrupado and treatment of missing or unclassified mode.'},
    {'decision_id': 'PENDING-05', 'topic': 'Matrix uncertainty', 'evidence_needed': 'Locate the reproducible variance or simulation method used for model-based EODH expansion and determine whether its inputs/replicates support uncertainty estimates by OD and mode cell.'},
    {'decision_id': 'PENDING-06', 'topic': 'Published benchmark comparability', 'evidence_needed': 'Establish whether the 16,723,044 daily trip benchmark uses the same record eligibility, geographic coverage, intrazonal treatment, and mode aggregation as the proposed matrix.'},
])

evidence_path = PROJECT_ROOT / 'docs/data_dictionary/eodh_methodology_evidence.csv'
pending_path = PROJECT_ROOT / 'docs/data_dictionary/eodh_matrix_open_decisions.csv'
evidence.to_csv(evidence_path, index=False, encoding='utf-8-sig')
pending.to_csv(pending_path, index=False, encoding='utf-8-sig')

print(f'Evidence ledger: {evidence_path.relative_to(PROJECT_ROOT)}')
print(f'Open decisions: {pending_path.relative_to(PROJECT_ROOT)}')
display(evidence)
display(pending)

Evidence ledger: docs\data_dictionary\eodh_methodology_evidence.csv
Open decisions: docs\data_dictionary\eodh_matrix_open_decisions.csv


,evidence_id,topic,documented_finding,source_path,page,evidence_status
0,EODH-01,Target population and geography,Private households with residents aged 5 or ol...,data/sources/raw/Datos tecnicos de la Encuesta...,20,documented_official
1,EODH-02,Statistical units,The observation unit is persons in sampled hou...,data/sources/raw/Datos tecnicos de la Encuesta...,20,documented_official
2,EODH-03,Travel reference period,Travel behavior refers to the day before the i...,data/sources/raw/contrato 2022-1972/Etapa IV/P...,13,documented_official_and_processing_report
3,EODH-04,Weighting and inference approach,The technical sheet describes a doubly robust ...,data/sources/raw/Datos tecnicos de la Encuesta...,11,documented_official
4,EODH-05,Final analytical record counts,"The final EODH database reports 22,755 househo...",data/sources/raw/contrato 2022-1972/Etapa IV/P...,127,published_aggregate_benchmark
5,EODH-06,Published expanded trip total,"The report gives 16,723,044 daily trips, with ...",data/sources/raw/contrato 2022-1972/Etapa IV/P...,128,published_aggregate_benchmark


,decision_id,topic,evidence_needed
0,PENDING-01,Trip eligibility,Map the official trip reference population and...
1,PENDING-02,Trip expansion factor,Find the field-level definition and operationa...
2,PENDING-03,ZAT geography,Verify zat_ori and zat_des code domains and th...
3,PENDING-04,Mode dimension,Confirm the published definition and domains o...
4,PENDING-05,Matrix uncertainty,Locate the reproducible variance or simulation...
5,PENDING-06,Published benchmark comparability,"Establish whether the 16,723,044 daily trip be..."


## 2. Confirm source fields and methodological boundary

The source schema confirms that the trip table contains `zat_ori`, `zat_des`, `modo_principal_agrupado`, and `fexp_vj`. The schema and analytical field catalog preserve source names but do not provide a documented definition for `fexp_vj`. No ZAT crosswalk or field-level factor derivation is established by this notebook.

The official design description is model-based and includes integration with EODH 2019 and population calibration. A conventional design-based variance estimator based on assumed clusters or strata would therefore be unsupported. Before reporting matrix uncertainty, the specific model-based variance or simulation procedure and its required inputs must be found and shown to apply to OD-by-mode cells.

The first OD products are planned as a complete daily matrix aggregated across modes plus separate daily matrices by the source main-mode grouping. Sample counts and expanded totals will be separate products. The expanded products remain blocked until the factor definition, eligible universe, and geography are validated.

In [3]:
readers = {
    'official_sheet': (OFFICIAL_SHEET, [10, 11, 20]),
    'processing_report': (PROCESSING_REPORT, [13, 127, 128]),
    'indicator_report': (INDICATOR_REPORT, [10, 18, 19]),
}
page_checks = []
for role, (relative_path, cited_pages) in readers.items():
    reader = PdfReader(PROJECT_ROOT / relative_path)
    for page_number in cited_pages:
        if page_number > len(reader.pages):
            raise ValueError(f'{role} has no page {page_number}: {relative_path.as_posix()}')
        page_checks.append({'source_role': role, 'source_path': relative_path.as_posix(), 'page': page_number, 'page_exists': True})

print('Cited page references verified:', len(page_checks))
display(pd.DataFrame(page_checks))
print('No microdata rows were read; no matrix or expansion was calculated.')

Cited page references verified: 9


,source_role,source_path,page,page_exists
0,official_sheet,data/sources/raw/Datos tecnicos de la Encuesta...,10,True
1,official_sheet,data/sources/raw/Datos tecnicos de la Encuesta...,11,True
2,official_sheet,data/sources/raw/Datos tecnicos de la Encuesta...,20,True
3,processing_report,data/sources/raw/contrato 2022-1972/Etapa IV/P...,13,True
4,processing_report,data/sources/raw/contrato 2022-1972/Etapa IV/P...,127,True
5,processing_report,data/sources/raw/contrato 2022-1972/Etapa IV/P...,128,True
6,indicator_report,data/sources/raw/documentation/a. Estimacion f...,10,True
7,indicator_report,data/sources/raw/documentation/a. Estimacion f...,18,True
8,indicator_report,data/sources/raw/documentation/a. Estimacion f...,19,True


No microdata rows were read; no matrix or expansion was calculated.


## Review gate

The evidence ledger and open-decision register are the outputs for review. The next step is to resolve the trip universe, source-zone mapping, and mode field contract, then construct the unweighted sample matrices. Do not implement weighted matrices or uncertainty estimates until the `fexp_vj` derivation and the official model-based inference method are supported by source documentation. EODI is not used in any EODH product.